<a href="https://colab.research.google.com/github/chungweichang0918/baseball-tactics-agent/blob/main/notebooks/colab_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 多模態棒球戰術 Agent：看影片、查 RE24、寫戰術報告
## Plan-and-Execute ＋ Tool Use ＋ Reflection

教練手上有一段 10–15 秒的投球或打擊影片，還有場上的局勢：幾局、幾出局、壘上有沒有人。這個 Agent 要回答的是：**這一球該不該盜壘、觸擊還是強攻？防守時要不要敬遠？**

流程分成五步：**偵察**（讀影片、讓 Gemini 看概況）→ **規劃**（Gemini 決定要用哪些工具）→ **執行**（YOLO 骨架、動作分段、球軌跡、RE24 查表）→ **撰寫**（Gemini 依證據寫報告）→ **審查**（規則檢查加 Gemini 檢查推論，不合格就退回修正）。

先說清楚它**不是**什麼：不是轉播級的比賽追蹤系統，單一手機鏡頭沒有相機校正，所以時間類數據比較可靠，球速、距離都只是粗估；也不是完全自主的 Agent，工具清單、停止條件與修正次數上限都由程式固定，Gemini 只在這個框架內做決定。

第 1、2 節準備環境，第 3 節直接打開 Gradio 工作台；第 4 節之後再由上往下，一段說明配一小格程式，把 Agent 拆開來看。影像分析的程式碼較大，放在專案的 `bbagent` 套件裡，這份筆記本逐步呼叫它的各個元件。建議使用 **T4 GPU** 執行階段（沒有 GPU 也能跑，骨架掃描比較慢）。本筆記本只產生報告，不會對外發布任何內容。

**怎麼執行**

上方選單「執行階段 → 全部執行」。第 3 節跑完，工作台就會顯示在那一格下方，同時產生公開連結；接著第 4 節會請你上傳一段影片，逐步示範 Agent 的每個環節。只想用工作台的話，上傳影片時按「取消」，逐步示範會停在那裡，工作台不受影響。

## 1. 取得專案程式碼

YOLO 骨架追蹤、動作分段、RE24 表等程式碼共有數千行，不適合全部貼進筆記本，所以先取得專案資料夾。三種方式擇一，填在下一格最上面：

| 方式 | 設定 |
|---|---|
| GitHub（預設） | `REPO_URL` 已填好本專案的 repo |
| Google Drive 公開連結（讓別人也能執行） | zip 設為「知道連結的人都能檢視」，`GDRIVE_FILE_ID` 填分享連結中 `/d/` 後面那一段 |
| 自己的 Google Drive | 兩者留空，把 `baseball-tactics-agent.zip` 放在「我的雲端硬碟」根目錄 |

In [ ]:
REPO_URL = "https://github.com/chungweichang0918/baseball-tactics-agent.git"
GDRIVE_FILE_ID = ""    # 例如 "1AbCdEf..."
DRIVE_ZIP = "/content/drive/MyDrive/baseball-tactics-agent.zip"

import os
import zipfile

PROJECT = "/content/baseball-tactics-agent"
if not os.path.exists(PROJECT):
    if REPO_URL:
        !git clone -q {REPO_URL} {PROJECT}
    else:
        if GDRIVE_FILE_ID:
            !gdown -q {GDRIVE_FILE_ID} -O /content/bbagent.zip
            zip_path = "/content/bbagent.zip"
        else:
            from google.colab import drive
            drive.mount("/content/drive")
            zip_path = DRIVE_ZIP
        with zipfile.ZipFile(zip_path) as z:
            z.extractall("/content")
%cd {PROJECT}

### 1-1. 安裝套件

固定版本，與開發時測試的版本相同。Colab 已內建有 CUDA 的 PyTorch，安裝時會沿用。若本次執行階段已載入其他版本的 Gradio，安裝後請重新啟動，再從下一格繼續。

安裝最後若出現 `google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0` 的紅字，可以忽略：這是 pip 的相容性提醒，不是安裝失敗。`google-genai` 需要較新的 `google-auth`（2.56 以上），不影響後面讀取 Secrets 與掛載 Drive。

In [ ]:
%pip install -q -e . "gradio==6.31.0" "google-genai==2.33.0" "ultralytics==8.4.192" "lap==0.5.13"

### 1-2. 載入模組、選擇裝置與模型

`pip install -e` 是「可編輯安裝」，要等 Python 重新啟動才會生效，所以這裡直接把專案的 `src` 資料夾加進匯入路徑，不必重新啟動執行階段。

`bbagent.vision.device` 必須在 PyTorch 之前載入（它設定 Apple MPS 的備援開關，在 Colab 上沒有副作用）。規劃、看圖、撰寫、審查共用 `config.yaml` 裡的同一個 Gemini 型號；它們的差別是收到的指令與資料不同，不是幾個各有人格的 AI。

In [ ]:
import os
import queue
import re
import sys
from getpass import getpass
from importlib.metadata import version
from pathlib import Path

SRC = Path("/content/baseball-tactics-agent/src")
if not SRC.exists():
    SRC = Path.cwd() / "src"
if not (SRC / "bbagent").exists():
    raise FileNotFoundError("找不到專案的 src/bbagent；請先執行第 1 節取得程式碼。")
os.chdir(SRC.parent)               # 重新啟動執行階段後，工作目錄會回到 /content
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from bbagent.vision import device  # 必須在 torch 之前

import gradio as gr
from IPython.display import HTML, Image, Markdown, display

from bbagent.config import load_config

print({name: version(name) for name in ["gradio", "google-genai", "ultralytics", "torch"]})
print("推論裝置：", device.resolve_device())
print("Gemini 型號：", load_config()["llm"]["fast_model"])

## 2. 連接 Gemini

在 Colab 左側鑰匙圖示新增名為 **`GEMINI_API_KEY`** 的密鑰，並開啟這份筆記本的存取權。沒有設定時改用隱藏輸入；直接按 Enter 留空，Agent 仍會執行，但規劃、看圖、撰寫與審查會改用不需要模型的預設流程。不要把金鑰寫進程式或截圖。

影片的幾張畫面、賽況與工具結果會送到 Gemini 處理，請使用可以公開的練習影片，不要上傳含有個資的內容。每次呼叫都可能消耗 API 額度。

In [ ]:
api_key = ""
try:
    from google.colab import userdata
    api_key = userdata.get("GEMINI_API_KEY") or ""
except Exception:
    pass
if not api_key.strip() and not os.environ.get("GEMINI_API_KEY"):
    api_key = getpass("請輸入 Gemini API key（輸入不會顯示；留空則不使用 Gemini）：")
if api_key.strip():
    os.environ["GEMINI_API_KEY"] = api_key.strip()
del api_key
print("已設定金鑰" if os.environ.get("GEMINI_API_KEY") else "未設定金鑰：Gemini 步驟會改用預設流程")

### 2-1. 診斷小幫手（不用背）

保留錯誤類別與訊息，遮蔽目前的金鑰與常見的 Google API 金鑰格式。這不是完整的個資過濾器，分享錯誤截圖前仍須檢查。

In [ ]:
def safe_error(error):
    detail = f"{type(error).__name__}: {error}"
    key = os.environ.get("GEMINI_API_KEY", "")
    if key:
        detail = detail.replace(key, "[API key 已遮蔽]")
    return re.sub(r"AIza[0-9A-Za-z_-]{20,}", "[API key 已遮蔽]", detail)[:2000]

### 2-2. 一次普通的模型呼叫

專案的 `GeminiClient` 包裝了三件事：遇到 429、5xx 與逾時會退避重試；同樣的輸入會重播快取，不重複消耗額度；結構化輸出會用 Pydantic 驗證。模型不會記得上一次 API 請求，後面每一步需要的資料都由程式重新放進 prompt。

Gemini 的重試訊息先放進 `notices` 佇列，稍後由事件小幫手印出來。

In [ ]:
from bbagent.llm.gemini_client import GeminiClient, GeminiError, model_name

notices = queue.Queue()
client = GeminiClient()
client.on_retry = lambda n, wait, reason: notices.put(f"Gemini 第 {n} 次重試：{reason}，{wait:g} 秒後再試")

### 2-3. 選做：先確認模型連線

預設不呼叫。只有需要診斷時才改成 `True`，會額外使用一次 API。

In [ ]:
RUN_CONNECTION_TEST = False
if RUN_CONNECTION_TEST:
    try:
        print(client.generate(model_name("fast"), ["請只回答：Gemini 連線成功"]).text)
    except GeminiError as e:
        print(safe_error(e))

### 2-4. 下載 YOLO 權重，準備事件小幫手

Agent 的每一步都會產生「事件」（AgentEvent）：思考、工具呼叫、工具結果、報告片段、審查意見。網頁介面與終端機都只讀這些事件。下面的 `show()` 把事件印出來，並取回該步驟的回傳值（例如 Planner 回傳的計畫）。

In [ ]:
!python scripts/download_models.py

ICONS = {"planner": "🧭", "executor": "🔧", "synthesizer": "✍️", "critic": "🔍", "system": "⚙️"}


def show(steps):
    """印出一個步驟產生的事件，回傳該步驟的結果。"""
    streaming = False
    while True:
        try:
            ev = next(steps)
        except StopIteration as stop:
            result = stop.value
            break
        if ev.kind == "report_delta":
            print(ev.content, end="", flush=True)
            streaming = True
            continue
        if streaming:
            print()
            streaming = False
        if ev.kind == "progress":
            print(f"   ⏳ {ev.content}")
        else:
            print(f"{ICONS[ev.stage]} {ev.title}")
            if ev.content and ev.kind != "final":
                print("   " + ev.content.replace("\n", "\n   "))
    while not notices.empty():
        print(f"   ⏳ {notices.get()}")
    return result

## 3. 先打開 Gradio 工作台

環境準備好了，先把完整的網頁工作台打開；第 4 節之後再把 Agent 拆開，一格一格看它怎麼運作。

### 3-1. 建立介面，尚未公開

工作台與 `scripts/app.py` 相同：上傳影片、選擇片段、填入賽況、指定投手與打者後分析，即時顯示計畫、工具呼叫、關鍵幀、動作曲線、報告與審查過程；守備頁是可以調整站位的情境模擬，不是影片追蹤結果。重新執行本格會先關閉舊介面。

In [ ]:
from bbagent.ui.app import build_app, launch_options

if "demo" in globals():
    demo.close()
demo = build_app()
print("介面已建立；下一格才啟動。")

### 3-2. 啟動 Colab 展示

`share=True` 會產生臨時公開連結：**取得連結的人可以透過介面上傳影片、使用你的 Gemini 額度**。金鑰不會顯示在畫面上，但這不代表別人不能使用介面。不要公開散布連結。

一次只處理一個分析，其他人會排隊。介面會直接顯示在這一格下方；公開連結只在這個 Colab 執行階段開著時有效。啟動後會立刻往下執行，不會卡在這一格；後面的逐步示範與工作台可以同時使用。不想公開時把 `START_DEMO` 改成 `False`。

In [ ]:
START_DEMO = True
if START_DEMO:
    demo.queue(max_size=8, default_concurrency_limit=1)
    demo.launch(share=True, inline=True, debug=False, show_error=False, **launch_options(demo))
else:
    print("尚未啟動；確認公開連結與 API 額度提醒後，將 START_DEMO 改成 True。")

## 4. 放入一段影片與賽況

專案資料夾裡有範例影片時直接使用；沒有時（例如從 GitHub 取得的版本不含影片）會請你上傳。想換成自己的影片，把 `UPLOAD_VIDEO` 改成 `True` 再執行。影片以 10–15 秒、包含 2–3 球（或 2–3 次揮棒）、固定機位的側拍最理想。

賽況是**我方觀點**：`score_diff` 是我方減對方，`our_role` 是我方在進攻或防守。`analysis_target` 留 `auto` 時由 Agent 判斷影片主角是投手還是打者。

In [ ]:
UPLOAD_VIDEO = False
VIDEO = Path("data/samples/p_other_01.mp4")
if UPLOAD_VIDEO or not VIDEO.exists():
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("沒有上傳影片，逐步示範到這裡停止；第 3 節的工作台不受影響，可以直接使用。")
    VIDEO = Path(next(iter(uploaded)))
print("影片：", VIDEO)

In [ ]:
from bbagent.schemas import GameState
from bbagent.ui.diagram import situation_svg

game = GameState(inning=7, half="bottom", outs=1, on_first=True, balls=0, strikes=0, score_diff=0,
                 our_role="offense", analysis_target="auto", pitcher_hand="R",
                 notes="一壘跑者腳程中上")
print(game.describe_zh())
display(HTML(situation_svg(game)))

## 5. 偵察：先看影片，再做計畫

規劃之前先做兩件固定的事：`probe_video` 讀解析度、fps、長度並偵測場景切換；`scout_overview` 把 4 張均勻取樣的畫面交給 Gemini，判斷拍攝視角、主角是投手還是打者、鏡頭有沒有移動。

每個工具的結果都會存成一筆**證據**，依序編號 E01、E02……。報告引用的每一個事實都必須指向其中一筆，Critic 會檢查。

In [ ]:
from bbagent.agent.executor import Executor
from bbagent.agent.orchestrator import default_run_dir
from bbagent.agent.state import AgentState
from bbagent.schemas import PlanStep

state = AgentState(video=VIDEO, game=game, run_dir=default_run_dir(VIDEO))
state.run_dir.mkdir(parents=True, exist_ok=True)
executor = Executor(client, notices)
show(executor.run_step(state, PlanStep(id="r1", tool="probe_video", purpose="讀取影片資訊")))
show(executor.run_step(state, PlanStep(id="r2", tool="scout_overview", purpose="判斷視角與拍攝條件")))
print("目前判斷的分析對象：", state.focus)

## 6. Planning：讓 Gemini 決定要用哪些工具

Planner 拿到賽況、偵察結果與**工具清單**（每個工具的用途、適用的投手或打者分支、前置工具），輸出一份結構化計畫：分析投手還是打者、要回答哪些戰術問題、評估哪些戰術、依序呼叫哪些工具。

### 6-1. 先看 Planner 收到什麼（不呼叫模型）

注意最後兩段：哪些戰術可以評估、哪些不行（例如兩出局不能犧牲觸擊、進攻方不能敬遠）是由程式依 RE24 規則算好，不交給模型猜。

In [ ]:
from bbagent.agent.planner import Planner, PlanDraft, plan_text, planner_input, validate_draft

print(planner_input(state))

### 6-2. 程式會檢查計畫（不呼叫模型）

Gemini 做決定，程式做檢查：工具是否存在、是否適用於這個分支、前置工具有沒有排在前面、必要步驟是否齊全、戰術是否合法。不通過時，錯誤訊息會附在 prompt 後面請 Gemini 重做，最多兩次；Gemini 無法使用時改用預設計畫，Agent 不會停擺。

下面手寫一份**故意寫錯**的計畫給檢查器看：投手分析卻漏了必要的指標工具，還混進打者專用的工具、在進攻方列入敬遠。

In [ ]:
bad = PlanDraft(analysis_focus="pitcher", rationale="示範用的錯誤計畫", tactical_questions=["要不要盜壘"],
                candidate_options=["steal_2b", "ibb"],
                steps=[{"id": "s1", "tool": "coarse_pose_scan", "purpose": "找人", "depends_on": []},
                       {"id": "s2", "tool": "segment_motions", "purpose": "分段", "depends_on": ["s1"]},
                       {"id": "s3", "tool": "swing_compare", "purpose": "比較揮棒", "depends_on": ["s2"]},
                       {"id": "s4", "tool": "re24_evaluate", "purpose": "查表", "depends_on": []}])
_, errors, warnings = validate_draft(bad, state)
print("錯誤：", *errors, sep="\n- ")
print("調整：", *warnings, sep="\n- ")

### 6-3. 第一次呼叫：賽況＋偵察 → 計畫

現在才呼叫 Gemini。執行後先讀計畫：選的分支對不對？戰術問題是否和賽況有關？`gemini_inspect` 的觀察重點是否具體？計畫可能需要修改，不是正確答案。

In [ ]:
planner = Planner(client)
plan = show(planner.plan(state))
state.plan = plan
print("\n計畫來源：", "Gemini 規劃" if state.plan_source == "llm" else "預設計畫")

## 7. Tool Use：依計畫執行工具

Executor 依相依順序（拓撲排序）呼叫工具，工具在背景執行緒中執行，YOLO 的進度與 Gemini 的重試會即時印出。除了 `scout_overview` 與 `gemini_inspect`，其他工具都是確定性的 Python 程式：

| 工具 | 做什麼 |
|---|---|
| `coarse_pose_scan` | YOLOv8 骨架模型＋ByteTrack 追蹤每個人，選出分析對象 |
| `camera_motion` | 用人物以外的背景特徵點量測鏡頭是否移動（OpenCV 光流） |
| `segment_motions` | 投手 6 個階段（固定姿勢→出手→收尾）或打者 5 個階段（準備→擊球→收棒） |
| `pitching_metrics`／`batting_metrics` | 到本壘時間（估）、抬腿高度、出手角度；或引棒到擊球時間、頭部移動量 |
| `track_ball` | 出手後逐幀找球、擬合軌跡，側拍時粗估球速 |
| `tipping_compare`／`swing_compare` | 比較各球或各次揮棒的差異 |
| `render_keyframes`、`gemini_inspect` | 畫關鍵幀，再請 Gemini 看骨架看不到的細節（視線、手套位置） |
| `re24_evaluate` | 查 RE24 得分期望值表，算每個戰術的 ΔRE 與損益平衡成功率 |

關鍵工具（例如找不到投球動作）失敗時，Planner 會排除失敗的工具與依賴它的工具，**重新規劃一次**。

In [ ]:
failure = show(executor.run_plan(state, plan))
if failure:
    plan2 = show(planner.replan(state, failure))
    if plan2.steps:
        show(executor.run_plan(state, plan2))

### 7-1. 看關鍵幀、動作曲線與球軌跡

綠框是分析對象，灰框是其他人；骨架左側畫藍色、右側畫橘色，方便檢查 YOLO 有沒有把左右標反。動作曲線上的直線是偵測到的動作階段。

In [ ]:
for k in state.runtime.get("keyframes", [])[:6]:
    print(k["caption"])
    display(Image(filename=k["path"], width=360))
for ev in state.evidence.values():
    if ev.tool in ("segment_motions", "track_ball"):
        for path in ev.result.artifacts:
            display(Image(filename=path, width=720))

### 7-2. 證據清單

這就是 Synthesizer 撰寫報告時拿到的全部事實。注意「部分成功」的證據：它們會帶著限制（例如非側拍、鏡頭移動），報告第 5 節必須交代。

In [ ]:
for ev in state.evidence.values():
    print(ev.brief(), end="\n\n")

## 8. 第二次呼叫：證據 → 報告

Synthesizer 收到賽況、計畫、證據清單與幾張關鍵幀，依固定範本串流撰寫報告：結論、影片觀察、RE24 數據、推論、信心與限制。每一項觀察都要引用證據編號；RE24 數字只能照抄工具算出的值。

In [ ]:
from bbagent.agent.synthesizer import Synthesizer

synthesizer = Synthesizer(client)
draft = show(synthesizer.write(state))

## 9. Reflection：Critic 審查，不合格就退回

審查分兩層。第一層是**規則**，便宜又精確：章節齊全、引用的證據真的存在、RE24 數字與工具結果一致、沒有建議違反賽況的戰術、提到洩漏球種時有用「疑似」、非側拍時有寫出限制。第二層才請 Gemini 檢查規則抓不到的**推論品質**：有沒有超出證據的宣稱、防守方有沒有把 ΔRE 看反。

### 9-1. 規則檢查（不呼叫模型）

先檢查剛才的報告，再把一個錯誤的 RE24 數字（ΔRE +0.999）塞進去，看規則能不能抓到。

In [ ]:
from bbagent.agent.critic_rules import check_rules
from bbagent.agent.orchestrator import inject_fault

print("原報告：", check_rules(state, draft) or "規則檢查通過")
for issue in check_rules(state, inject_fault(draft)):
    print(f"注入錯誤後：[{issue['severity']}] {issue['location']}：{issue['message']}")

### 9-2. 完整審查，必要時退回修正

`Critic.review()` 先跑規則，全部通過才呼叫 Gemini。只要有一項 high 等級的問題就退回，Synthesizer 會帶著審查意見（含原文片段）重寫，最多兩次。修正上限是刻意設的停止條件：審查意見不一定都對，無限迴圈只會耗光額度。

In [ ]:
from bbagent.agent.critic import Critic

critic = Critic(client)
for round_no in range(3):
    verdict = show(critic.review(state, draft, round_no))
    if verdict.passed or round_no == 2:
        break
    feedback = [f"{i['location']}：{i['message']}" for i in verdict.blocking]
    draft = show(synthesizer.write(state, feedback, draft, round_no + 1))
display(Markdown(draft))

## 10. 這個 Agent 用了哪些設計模式？

**Plan-and-Execute**：先由模型產生完整計畫，再逐步執行。和 ReAct「想一步、做一步、看結果再想下一步」不同，計畫是一份可以讀、可以檢查的中間成果，也比較省模型呼叫；代價是執行中途的新資訊只能透過「重新規劃」回饋。[Wang 等，2023](https://arxiv.org/abs/2305.04091)、[Yao 等，2023](https://arxiv.org/abs/2210.03629)

**Tool Use**：模型不自己算數字。骨架、時間、球速、RE24 都交給確定性的程式，模型只負責「選工具」與「解讀證據」。報告裡的每個數字都能追溯到一筆證據。

**Reflection**：先寫、再審、依審查意見修正。這裡的審查一半是程式規則、一半是模型；規則保證數字與格式不出錯，模型負責判斷推論是否誇大。[Madaan 等，2023](https://arxiv.org/abs/2303.17651)、[Shinn 等，2023](https://arxiv.org/abs/2303.11366)

**RE24**：在每一種「壘包 × 出局數」的局面下，該半局平均還能得幾分。戰術的價值是局面改變前後的差（ΔRE），損益平衡成功率則是盜壘這類有成功與失敗兩種結果的戰術，成功率要多高才划算。[Tom Tango 的 MLB 2010–2015 表](https://tangotiger.net/re24.html)

這仍然不是完全自主的 Agent：工具清單、檢查規則、重新規劃一次、修正兩次都是程式寫死的停止條件。

## 11. 最後把全部包成一個函式

前面逐格看過每個元件，`run_agent()` 只是把它們串起來，依序產生同樣的事件，並把報告與執行紀錄（每一步的事件、證據與審查）存在 `runs/agent/` 底下。`demo_fault=True` 會在第一版報告刻意注入錯誤數字，用來展示 Critic 退回修正。

預設不重跑，改成 `True` 會再完整執行一次（數次 Gemini 呼叫；骨架掃描結果有快取，不會重算）。

In [ ]:
RUN_PIPELINE_TEST = False
if RUN_PIPELINE_TEST:
    from bbagent.agent.orchestrator import run_agent

    final = None
    for ev in run_agent(VIDEO, game, client=client, demo_fault=True):
        if ev.kind in ("critique", "final"):
            print(f"{ICONS[ev.stage]} {ev.title}")
        final = ev
    display(Markdown(final.content))

## 12. 展示完畢再停止

展示期間不要執行這格。結束後改成 `True` 執行，或中斷並刪除執行階段，避免留下仍可使用額度的服務。

In [ ]:
STOP_DEMO = False
if STOP_DEMO:
    demo.close()
    print("展示服務已停止。")

## 13. 這份作業要帶走什麼？

Agent 的重點不是讓模型做所有事，而是**讓模型做判斷、讓程式做計算與檢查，並把每一步的中間成果攤開**：計畫可以讀、證據有編號、報告的數字能追溯、審查意見會寫出哪一句有問題。

可以做三個比較：
1. 用 `demo_fault=True` 跑一次，看 Critic 擋下錯誤數字、退回修正的過程。規則抓得到哪些錯？抓不到哪些？
2. 同一段影片把賽況改成兩出局，或改成我方防守，Planner 選的工具與候選戰術怎麼變？
3. 換一段非側拍、鏡頭會動的影片，報告的「信心與限制」有沒有誠實交代？

### 參考資料與驗證界線

- [Plan-and-Solve Prompting](https://arxiv.org/abs/2305.04091)、[ReAct](https://arxiv.org/abs/2210.03629)：規劃與逐步行動兩種 Agent 模式。
- [Self-Refine](https://arxiv.org/abs/2303.17651)、[Reflexion](https://arxiv.org/abs/2303.11366)：以回饋修正輸出的 Reflection 模式。
- [Tom Tango RE24 表](https://tangotiger.net/re24.html)（資料來源 Retrosheet）：本專案使用 MLB 2010–2015 的數值。2023 年起的規則改變（投手計時器、牽制次數限制、加大壘包）使盜壘成功率上升，這張表沒有反映。
- [Ultralytics YOLO 文件](https://docs.ultralytics.com/)：骨架模型與 ByteTrack 追蹤。
- [Google Gen AI SDK](https://googleapis.github.io/python-genai/)：結構化輸出與串流。

動作分段的門檻是用合成資料與少量真實影片調整的；手機影片中的球常只有幾個像素，找不到軌跡是常態。單元測試可以檢查程式、資料傳遞與錯誤處理，不能證明真實模型的報告品質，仍須用自己的金鑰與影片在 Colab 觀察。

整理日期：2026-10-30。本檔不保存 API key 或執行輸出。